# Save frozen CSA semantic banks — exact Output130
Accelerator **None**, Internet **ON**. No Add Input. Enable Secret `KAGGLE_API_TOKEN`; if named `codex`, change SECRET_NAME below. Never paste the token in code/chat.
This only downloads and saves the completed cache. No CLIP/CSA rerun. New Private Dataset; preserve Output130 until processing finishes.


In [ ]:
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/130'
OWNER='ducanh2403'
SLUG='csa-semantic-banks-v130'
TITLE='CSA semantic banks v130 dataset'
SECRET_NAME='KAGGLE_API_TOKEN'
DESTINATION=Path('/kaggle/working/csa_semantic_banks_v130_dataset')


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/130':raise ValueError('This notebook pins130; do not substitute latest or another version.')
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('TITLE/SLUG must have6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/130', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v130', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v130; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import hashlib,json
INVENTORY={'kernel_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/130', 'files': {'semantic_banks.npz': 'e77608563fb23622d52d27ada06f49e6e578ef573c9fe41861878c36357fa698', 'cache_manifest.json': '1fe9b24391bc580acf13cf488224e92a187389d680fa40293d0f22ce15b1d8ea', 'cache_receipt.json': 'a50d8a43ab4eb0f69f1bf91644054c4a6b30da65cb54194f804815793868de24', 'smoke_summary.json': 'd59ffc2496f71ac5c65c995ab19d60affa28865c58c71eb624bb9257ebaefc90', 'notebook.log': '2c0bb4ffab63bad3993a379797d05583472bdffa7ee2c99c4df5feaf1d7d20ca', 'source_receipt.json': '5e26da5acd1d597441e0497763ef3c0afcf0e19e90731e9a2ad8717afc75c537'}, 'source_review_zip_sha256': '7b14d35af654b630f6a5addf9943dbe5910074c9f772e7d8ee38026558f15f9e'}
SOURCE_CODE='"""Byte-preserving cache transport; no API/model imports."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef package(source,destination,inventory):\n    source,destination=Path(source),Path(destination)\n    if inventory[\'kernel_version\']!=\'ducanh2403/2-vi-cai-kia-qua-nhieu/130\':raise ValueError(\'Expected exact Output130\')\n    if destination.exists():raise ValueError(\'Preserve existing destination; use a fresh path\')\n    for name,h in inventory[\'files\'].items():\n        if Path(name).name!=name or not (source/name).is_file() or sha(source/name)!=h:raise ValueError(\'Missing/changed output: \'+name)\n    read=lambda name:json.loads((source/name).read_text(encoding=\'utf-8\'))\n    receipt=read(\'cache_receipt.json\');smoke=read(\'smoke_summary.json\');manifest=read(\'cache_manifest.json\')\n    if receipt[\'status\']!=\'SEMANTIC_BANK_CACHE_COMPLETE\' or smoke[\'status\']!=\'REAL_INPUT_SMOKE_COMPLETE\':raise ValueError(\'Incomplete source\')\n    for name,h in receipt[\'payload_sha256\'].items():\n        if inventory[\'files\'].get(name)!=h:raise ValueError(\'Cache receipt payload differs\')\n    if manifest[\'source_bank_sha256\']!=receipt[\'inputs\'][\'overlay_sha256\'][\'bank_words_corrected.json\']:raise ValueError(\'Source bank differs\')\n    destination.mkdir(parents=True)\n    for name,h in inventory[\'files\'].items():\n        shutil.copyfile(source/name,destination/name)\n        if sha(destination/name)!=h:raise ValueError(\'Copied bytes differ: \'+name)\n    report=dict(status=\'SEMANTIC_BANK_V130_TRANSPORT_COMPLETE\',source_kernel_version=inventory[\'kernel_version\'],\n        source_review_zip_sha256=inventory[\'source_review_zip_sha256\'],files_sha256=inventory[\'files\'],\n        original_bytes_preserved=True,model_executed=False,also_requires_dataset=\'ducanh2403/csa-real-v126-lemma-fixed\')\n    (destination/\'transport_receipt.json\').write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return report\n'
assert hashlib.sha256(SOURCE_CODE.encode()).hexdigest()=='13dd68656a6cb0405dc45dc3b6c5b55ec5124ee0b8eaec99c0265e88d90a6b41'
exec(compile(SOURCE_CODE,'package_bank_dataset.py','exec'))


In [ ]:
import tempfile,shutil
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<256*1024**2:
    raise RuntimeError('Need256MiB free; preserve existing files.')
DOWNLOAD=Path(tempfile.mkdtemp(prefix='csa-bank-v130-',dir='/tmp'))
pattern=r'(^|/)(?:'+'|'.join(re.escape(n) for n in INVENTORY['files'])+r')$'
print('Downloading EXACT Output:',KERNEL_VERSION,flush=True)
run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),'--file-pattern',pattern])
matches=[p.parent for p in DOWNLOAD.rglob('cache_receipt.json') if sha(p)==INVENTORY['files']['cache_receipt.json']]
if len(matches)!=1:raise RuntimeError(f'Output130 returned {len(matches)} matching receipts. STOP; no latest fallback.')
report=package(matches[0],DESTINATION,INVENTORY)
print('All six original payloads verified/copied unchanged.',flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private research cache from exact notebook Output130: frozen CLIP512D vectors for5269nouns/1594verbs and provenance. Upstream model/data terms retained. No trained caption checkpoint. Original ROI/vocabulary remain in csa-real-v126-lemma-fixed.')
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; choose a fresh destination.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('Creation accepted; Dataset may still be processing. Do not repeatedly create.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','status',f'{OWNER}/{SLUG}'])
except RuntimeError:print('Status unconfirmed; check Dataset page before retrying.',flush=True)


Send the actual mount after adding the completed Dataset.
Expected `/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130`.
Keep the earlier `csa-real-v126-lemma-fixed` Dataset too: it contains ROI features/vocabulary needed for training.
